In [1]:
from pathlib import Path
import pandas as pd

wines_features = pd.read_csv(Path("../data/processed/wines_features.csv"))
wines_features.head()

,WineID,WineName,Type,Elaborate,Grapes,Harmonize,ABV,Body,Acidity,Country,...,Acidity_nat,Country_nat,RegionName_nat,WineryName_nat,Grapes_nat,Harmonize_nat,ABV_bin,ABV_nat,content_text,bert_text
0,100001,Espumante Moscatel,Sparkling,Varietal/100%,['Muscat/Moscato'],"['Pork', 'Rich Fish', 'Shellfish']",7.5,Medium-bodied,High,Brazil,...,high,brazil,serra gaúcha,casa perini,['muscat moscato'],"['pork', 'rich fish', 'shellfish']",abv_low,7.5% abv,type_sparkling elaborate_varietal_100 body_med...,espumante moscatel. This wine is a medium bodi...
1,100002,Ancellotta,Red,Varietal/100%,['Ancellotta'],"['Beef', 'Barbecue', 'Codfish', 'Pasta', 'Pizz...",12.0,Medium-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,casa perini,['ancellotta'],"['beef', 'barbecue', 'codfish', 'pasta', 'pizz...",abv_12_13,12.0% abv,type_red elaborate_varietal_100 body_medium_bo...,"ancellotta. This wine is a medium bodied, medi..."
2,100003,Cabernet Sauvignon,Red,Varietal/100%,['Cabernet Sauvignon'],"['Beef', 'Lamb', 'Poultry']",12.0,Full-bodied,High,Brazil,...,high,brazil,serra gaúcha,castellamare,['cabernet sauvignon'],"['beef', 'lamb', 'poultry']",abv_12_13,12.0% abv,type_red elaborate_varietal_100 body_full_bodi...,cabernet sauvignon. This wine is a full bodied...
3,100004,Virtus Moscato,White,Varietal/100%,['Muscat/Moscato'],['Sweet Dessert'],12.0,Medium-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,monte paschoal,['muscat moscato'],['sweet dessert'],abv_12_13,12.0% abv,type_white elaborate_varietal_100 body_medium_...,"virtus moscato. This wine is a medium bodied, ..."
4,100005,Maison de Ville Cabernet-Merlot,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']","['Beef', 'Lamb', 'Game Meat', 'Poultry']",11.0,Full-bodied,Medium,Brazil,...,medium,brazil,serra gaúcha,aurora,"['cabernet sauvignon', 'merlot']","['beef', 'lamb', 'game meat', 'poultry']",abv_11_12,11.0% abv,type_red elaborate_assemblage_bordeaux_red_ble...,maison de ville cabernet merlot. This wine is ...


In [2]:
wines_features = wines_features[wines_features["content_text"].notna()].copy()
wines_features = wines_features[wines_features["content_text"].str.strip() != ""].copy()

print(wines_features.shape)

(100646, 36)


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import pandas as pd

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(wines_features["content_text"])

# reset index so row positions match matrix rows
wines_features = wines_features.reset_index(drop=True)

wineid_to_idx = pd.Series(wines_features.index, index=wines_features["WineID"]).drop_duplicates()

In [6]:
def recommend_similar_wines(wine_id, top_k=5):
    if wine_id not in wineid_to_idx:
        raise ValueError(f"WineID {wine_id} not found.")
    
    idx = wineid_to_idx[wine_id]
    
    # similarity of one wine against all wines
    sim_scores = linear_kernel(tfidf_matrix[idx:idx+1], tfidf_matrix).flatten()
    
    # get top_k + 1 because the first one is the wine itself
    top_indices = sim_scores.argsort()[::-1][1:top_k+1]
    
    results = wines_features.loc[top_indices, [
        "WineID", "WineName", "Type", "Country", "RegionName"
    ]].copy()
    
    results["similarity_score"] = sim_scores[top_indices]
    return results

In [7]:
sample_wine_id = wines_features.iloc[0]["WineID"]
recommend_similar_wines(sample_wine_id, top_k=5)

,WineID,WineName,Type,Country,RegionName,similarity_score
1479,101480,Espumante Moscatel,Sparkling,Brazil,Serra Gaúcha,0.650996
415,100416,Ice,Sparkling,Brazil,Serra Gaúcha,0.589593
638,100639,Macaw Tropical Moscato Frisante Rosé,Sparkling,Brazil,Serra Gaúcha,0.576104
537,100538,Macaw Tropical Frisante Branco,Sparkling,Brazil,Serra Gaúcha,0.576104
429,100430,Sauvignon Blanc,White,Brazil,Serra Gaúcha,0.541617


In [9]:
sample_ids = wines_features["WineID"].sample(3, random_state=42).tolist()

for wine_id in sample_ids:
    print(f"\nRecommendations for WineID {wine_id}")
    display(recommend_similar_wines(wine_id, top_k=5))


Recommendations for WineID 121539


,WineID,WineName,Type,Country,RegionName,similarity_score
32620,132642,Les Grandes Terres Rasteau,Red,France,Rasteau,0.655371
18121,118143,Les Hauts du Village Rasteau,Red,France,Rasteau,0.655371
35062,135084,Les Héritiers Rasteau,Red,France,Rasteau,0.655371
13335,113357,Tradition Rasteau,Red,France,Rasteau,0.655371
13617,113639,Prestige Rasteau,Red,France,Rasteau,0.647197



Recommendations for WineID 169173


,WineID,WineName,Type,Country,RegionName,similarity_score
69208,169314,Don Nicanor Chardonnay-Viognier,White,Argentina,Mendoza,0.608018
69703,169809,Santa Isabel Chardonnay-Viognier,White,Argentina,Mendoza,0.589216
68337,168443,Chardonnay,White,Argentina,Mendoza,0.575517
69489,169595,Reserva Chardonnay,White,Argentina,Mendoza,0.521722
69376,169482,Extra Brut,Sparkling,Argentina,Mendoza,0.496092



Recommendations for WineID 198191


,WineID,WineName,Type,Country,RegionName,similarity_score
95694,195823,Hommage Cabernet Sauvignon,Red,Israel,Judean Hills,0.604711
98711,198840,Hommage Gewürztraminer,White,Israel,Judean Hills,0.595452
86726,186838,Hommage Chardonnay,White,United States,Calistoga,0.421481
96024,196153,Viognier,White,Israel,Judean Hills,0.386621
98649,198778,White Sauvignon Blanc-Chardonnay,White,Israel,Judean Hills,0.377719
